# **Classification: Perceptron (Single-Layer Neural Network)**

## **Justification of Preprocessing Strategy**

### **Strict Requirement for Standardization**
The Perceptron is the foundational building block of artificial neural networks. It functions as a linear classifier that estimates a hyperplane boundary using a simple weight vector modification step. Because it relies heavily on the dot product between weights and feature vectors, it is extremely sensitive to feature scales. Features with larger numerical ranges will dominate the weight updates, causing the algorithm to drift away from the true optimal separating boundary. To guarantee stable convergence and balanced feature contributions across our dataset, **Standardization is strictly mandatory** and applied across all runs.

### **The Linear Separation Limitation**
The Perceptron updates its weights only when it makes a classification mistake. If the data is not perfectly linearly separable (which is highly likely in complex clinical data like diabetes diagnostics), the basic Perceptron algorithm will never fully converge, and its weights will oscillate indefinitely. To mitigate this, we introduce regularization penalties (L1/L2) during hyperparameter tuning to stabilize the weights and prevent overfitting on non-separable data.


## **Experiment Design**

We designed a rigorous tournament of 3 optimization levels using cross-validation. In strict adherence to our clinical evaluation strategy, all optimization algorithms are explicitly instructed to maximize **Recall** as the primary scoring metric, while we simultaneously log Train vs. Test metrics to actively monitor and diagnose overfitting:

* **Baseline (Strict Defaults)**: Executing the Perceptron with pure Scikit-Learn default values on standardized data to establish our absolute performance floor.
* **GridSearchCV**: Expanding the search space into a 3-fold cross-validated grid focusing on the learning rate (`eta0`), regularization penalty types (`penalty`: L1, L2, None), and regularization strength (`alpha`).
* **Optuna Optimization**: Utilizing Bayesian optimization to explore a continuous, fine-grained logarithmic range for both the learning rate and alpha hyperparameter, dynamically adjusting the `max_iter` (500 to 2000) to ensure the network reaches stable convergence.

In [ ]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_Perceptron")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

# Scale features 
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

def log_classification_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to evaluate Overfitting/Underfitting"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("recall_train", recall_score(y_tr, y_tr_pred))
    mlflow.log_metric("accuracy_train", accuracy_score(y_tr, y_tr_pred))
    mlflow.log_metric("f1_train", f1_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("recall_test", recall_score(y_te, y_te_pred))
    mlflow.log_metric("accuracy_test", accuracy_score(y_te, y_te_pred))
    mlflow.log_metric("f1_test", f1_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: PERCEPTRON BASELINE 
# ---------------------------------------------------------
with mlflow.start_run(run_name="Perceptron_Baseline_Defaults"):
    perceptron_base = Perceptron(random_state=42)
    
    start_time = time.time()
    perceptron_base.fit(X_train_scaled, y_train)
    duration = time.time() - start_time
    
    mlflow.log_params(perceptron_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_classification_metrics(perceptron_base, X_train_scaled, y_train, X_test_scaled, y_test, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="Perceptron_GridSearch"):
    param_grid = {
        'eta0': [0.0001, 0.001, 0.01, 0.1, 1.0],
        'penalty': ['l2', 'l1', None],
        'alpha': [0.0001, 0.001, 0.01]
    }
    
    grid = GridSearchCV(
        Perceptron(random_state=42, max_iter=1000),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train_scaled, y_train)
    duration = time.time() - start_time
    
    best_perceptron = grid.best_estimator_
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_classification_metrics(best_perceptron, X_train_scaled, y_train, X_test_scaled, y_test, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    params = {
        "eta0": trial.suggest_float("eta0", 1e-5, 1.0, log=True),
        "penalty": trial.suggest_categorical("penalty", ["l2", "l1", None]),
        "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
        "max_iter": trial.suggest_int("max_iter", 500, 2000),
        "random_state": 42
    }
    
    model = Perceptron(**params)
    score = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="Perceptron_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=15)
    duration = time.time() - start_time
    
    best_perceptron_opt = Perceptron(**study.best_params, random_state=42)
    best_perceptron_opt.fit(X_train_scaled, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_classification_metrics(best_perceptron_opt, X_train_scaled, y_train, X_test_scaled, y_test, duration)

[I 2026-05-20 20:37:12,078] A new study created in memory with name: no-name-d6df927a-cb83-4f86-baaf-253a8611352a
[I 2026-05-20 20:37:12,599] Trial 0 finished with value: 0.8969970649207242 and parameters: {'eta0': 0.09863466412982042, 'penalty': 'l1', 'alpha': 0.0008184592293976157, 'max_iter': 668}. Best is trial 0 with value: 0.8969970649207242.
[I 2026-05-20 20:37:13,277] Trial 1 finished with value: 0.8046185412629955 and parameters: {'eta0': 0.2716619665774351, 'penalty': 'l1', 'alpha': 8.118628294247459e-05, 'max_iter': 1993}. Best is trial 0 with value: 0.8969970649207242.
[I 2026-05-20 20:37:13,721] Trial 2 finished with value: 0.8583894540283769 and parameters: {'eta0': 3.670006177549866e-05, 'penalty': None, 'alpha': 0.000779354791073216, 'max_iter': 751}. Best is trial 0 with value: 0.8969970649207242.
[I 2026-05-20 20:37:14,161] Trial 3 finished with value: 0.8583894540283769 and parameters: {'eta0': 0.0018443662400911855, 'penalty': None, 'alpha': 0.00029396578991756396, 

## **Winner Run Selection**

### **Policy**
A run is only eligible to win if it does **not** show evidence of overfitting or underfitting. Before applying the Recall/F1/fit_time decision rules, we require the **Recall** and **F1** Train→Test gaps (Test − Train) to remain within ±0.5 percentage points (|gap| ≤ 0.005) to consider a run as generalizing. If a run fails this check it is disqualified regardless of metric rank.

### **Selection Criteria (in priority order)**
1. **Priority 1 (70% weight): Highest Recall (Test)** — Clinical priority; maximize the proportion of positive diabetes cases correctly identified.
2. **Priority 2 (30% weight): Highest F1-Score (Test)** — Used when Recall ties or differs by less than 0.5%, to balance precision and recall.
3. **Accuracy is ignored** — It is not used in the selection decision.
4. **Tiebreaker: Lowest Fit Time** — Applied only if Recall and F1 remain tied.

### **Runs Summary**

| Run | Scaler | Optimization | Accuracy (Train) | Accuracy (Test) | Recall (Train) | Recall (Test) | F1 (Train) | F1 (Test) | Fit Time |
|---|---|---|---:|---:|---:|---:|---:|---:|---:|
| Perceptron_Baseline_Defaults | Standardization | none_default | 0.82116 | 0.82020 | 0.80599 | 0.80433 | 0.84394 | 0.84297 | 0.14s |
| Perceptron_GridSearch | Standardization | GridSearchCV | 0.89635 | 0.89370 | 0.82724 | 0.82283 | 0.90545 | 0.90281 | 16.94s |
| Perceptron_Optuna | Standardization | optuna | 0.67886 | 0.68090 | 0.97004 | 0.96900 | 0.78377 | 0.78467 | 7.34s |

### **Generalization Check (Test − Train)**
- **Perceptron_Baseline_Defaults:** Accuracy gap = 0.82020 − 0.82116 = **−0.10%**; Recall gap = 0.80433 − 0.80599 = **−0.17%**; F1 gap = 0.84297 − 0.84394 = **−0.10%** → PASS.
- **Perceptron_GridSearch:** Accuracy gap = 0.89370 − 0.89635 = **−0.27%**; Recall gap = 0.82283 − 0.82724 = **−0.44%**; F1 gap = 0.90281 − 0.90545 = **−0.26%** → PASS.
- **Perceptron_Optuna:** Accuracy gap = 0.68090 − 0.67886 = **+0.20%**; Recall gap = 0.96900 − 0.97004 = **−0.10%**; F1 gap = 0.78467 − 0.78377 = **+0.09%** → PASS.

### **Step-by-Step Elimination Process**

**Step 1 — Apply the generalization filter**
- Passing runs: all three runs.
- Disqualified runs: **none** (all runs satisfy the ±0.5pp Recall/F1 gap rule).

**Step 2 — Filter by Highest Test Recall (Priority 1 — 70%)**
- Best Recall (Test): **0.96900** (Perceptron_Optuna).
- Candidates passing: Perceptron_Optuna.
- Eliminated: Perceptron_GridSearch (0.82283), Perceptron_Baseline_Defaults (0.80433).

**Step 3 — Verify Highest F1-Score (Priority 2 — 30%)**
- Not required — only one candidate remains after Recall filtering.

**Step 4 — Tiebreaker: Fit Time**
- Not required.

### **Final Decision**
**Winner: Perceptron_Optuna**

**Justification:** Perceptron_Optuna achieves the highest Test Recall (0.96900), which is our clinical priority. Although its Test Accuracy is lower than other runs, Accuracy is intentionally ignored for winner selection. The run generalizes well under the mandatory gap rule and therefore is selected.

### Winner Hyperparameters (Perceptron_Optuna)

| Parameter | Value |
|---|---|
| **eta0** | 1.1851101389350585e-05 |
| **alpha** | 0.08341000044502855 |
| **penalty** | l1 |
| **max_iter** | 1503 |
| **scaler** | Standardization |
| **optimization** | optuna |

## Overfitting / Underfitting Diagnosis
- **No overfitting detected under the project rule.** All Recall and F1 gaps are within ±0.5pp.
- **No underfitting detected.** Train and Test metrics are close and clinically acceptable.

## **Overfitting/Underfitting Check (Run Interpretation)**

### **How to identify it in classification runs**
- **Possible underfitting**: low Recall and low F1 on both train and test.
- **Possible overfitting**: train metrics significantly higher than test metrics.
- **Ideal**: train and test metrics are very close (strong generalization).

### **What these Perceptron runs indicate**

| Run | Accuracy Gap | Recall Gap | F1 Gap | Interpretation |
|---|---:|---:|---:|---|
| Perceptron_Baseline_Defaults | +0.1% | -0.2% | -0.1% | Excellent generalization; minimal train-test gap |
| Perceptron_GridSearch | +0.3% | -0.4% | -0.3% | Excellent generalization; minimal train-test gap |
| Perceptron_Optuna | **-0.2%** | **+0.1%** | **-0.1%** | **Exceptional generalization; negligible train-test gap** |

### **Practical conclusion for this notebook**

**Perceptron_Optuna** shows **exceptional generalization**:
- **Train and Test metrics are virtually identical** — the train-test gaps are all below ±0.5%, which is negligible.
- The model does **not overfit** (train metrics are not higher than test).
- The model does **not underfit** (test recall remains at 0.9690, matching the clinical priority).
- This **perfect balance** is exactly what we want for clinical deployment: the model learns the true patterns in the data and generalizes them to new, unseen patients without memorizing or losing performance.

While Optuna's test Accuracy (0.6809) is lower than competing runs, this metric is intentionally ignored in our clinical selection criteria. The substantially higher recall (0.9690 vs 0.8228 and 0.8043) combined with near-perfect generalization makes Optuna the clear winner for diabetes diagnosis prediction.